In [1]:
import sys
sys.path.append('../utils_function/')
import utils_function as uf
import common_functions as cf
import pandas as pd
import numpy as np
import os
from collections import defaultdict

In [2]:
def get_optimal_params(performance_dict):
    """
    从性能字典中找出平均相关性最高的参数
    
    参数:
    performance_dict (dict): 包含不同参数对应相关性列表的字典
    
    返回:
    float: 最优参数值
    """
    best_avg_corr = -float('inf')  # 初始化为负无穷

    for param, corr_list in performance_dict.items():
        avg_corr = sum(corr_list) / len(corr_list)
        std_corr = np.std(corr_list)
        # print(f"参数: {param}, 平均相关性: {avg_corr:.4f}, 标准差: {std_corr:.4f}")
        
        # 更新最优参数
        if avg_corr > best_avg_corr:
            best_avg_corr = avg_corr
            best_param = param

    return best_param

In [3]:
# animal_list = ['mouse', 'marmoset', 'macaque', 'human', 'marmoset']
# dataset_key_list = ['gamanut', 'majka', 'markov40', 'lariviere', 'V4']
animal_list = ['mouse', 'marmoset', 'macaque', 'human']
dataset_key_list = ['gamanut', 'majka', 'markov40', 'lariviere']
performance_dict = {}
optimal_params_dict = {}
for animal, dataset_key in zip(animal_list, dataset_key_list):
    print(f'Running for {animal} and {dataset_key}')

    performance_dict_this_animal = defaultdict(list)

    model_container = cf.MetaModelContainer(uf.AxonGrowthModel, cf.pj('../../results/para_search_adapted_betzel_like', f'{animal}_{dataset_key}'))

    for model in model_container:
        performance_dict_this_animal[(model.Lambda, model.gamma)].append(model.get_data_model_correlation())
    print(performance_dict_this_animal)
    performance_dict[f'{animal}_{dataset_key}'] = performance_dict_this_animal
    best_Lambda, best_gamma = get_optimal_params(performance_dict_this_animal)
    optimal_params_dict[f'{animal}_{dataset_key}'] = {'Lambda': best_Lambda, 'gamma': best_gamma}
    print(performance_dict_this_animal)
    print(f'Optimal lambda for {animal}_{dataset_key}: {optimal_params_dict[f"{animal}_{dataset_key}"]}')
    print(f'Optimal average correlation: {max([sum(v)/len(v) for v in performance_dict_this_animal.values()])}')

Running for mouse and gamanut
defaultdict(<class 'list'>, {(0.22407790126453542, 1.0): [0.42243017029266267], (0.4101947531613386, 1.4000000000000004): [0.39286579251493253], (0.7824284569549448, 1.0): [0.31614001131571107], (0.6273310803742755, 2.0): [0.346834078873146], (0.8754868829033464, 2.0): [0.3056638257080489], (0.5963116050581417, 1.8000000000000003): [0.35255001258069163], (0.9375258335356141, -0.7999999999999998): [0.1047112495854329], (0.4412142284774724, -0.19999999999999996): [0.2826184591590764], (0.2550973765806693, 1.2000000000000002): [0.42035147071210904], (0.2550973765806693, 2.0): [0.4209139103592134], (0.317136327212937, -0.3999999999999999): [0.311218203561233], (0.8444674075872125, -0.19999999999999996): [0.14944545749245683], (0.7203895063226771, 0.40000000000000036): [0.3259559669955626], (0.1, 0.20000000000000018): [0.4647950425900588], (0.9375258335356141, 0.8000000000000003): [0.2935496171415856], (0.8444674075872125, 0.0): [0.5127894429650411], (0.9685453

In [4]:
if cf.check_all_file_exist_with_any_extension('../../results/optimal_params_adapted_betzel_like/optimal_params_adapted_betzel_like'):
    print('already saved, please check whether you want to overwrite it')
else:
    cf.save_dict(optimal_params_dict, '../../results/optimal_params_adapted_betzel_like/optimal_params_adapted_betzel_like')

already saved, please check whether you want to overwrite it
